# 4-2절 연습 문제 풀이

이 노트북은 4-2절 연습 문제(4-4, 4-5)의 풀이 예시다.

- 본문 예제 코드는 `code_examples/ch04/04-02_example.ipynb`를 참고한다.
- 4-4는 서술형 문제이므로 해설만 싣고, 4-5는 구현과 실행 결과를 함께 싣는다.
- 각 문제마다 **풀이 해설**과 **문제 검토**를 함께 실었다.

In [1]:
import copy
import csv
import random

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split, ConcatDataset

SEED = 1
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DATA_DIR = '../../data'
LR = 0.01
HIDDEN_DIM = 64

## 연습 문제 4-4

> 과적합을 확인하기 위해서는 훈련 데이터셋과 검증, 평가 데이터셋을 반드시 분리해야 한다.
> 그렇다면 검증 데이터셋과 평가 데이터셋을 따로 분리하는 이유는 무엇인지 유추해 보자.
>
> 힌트: 검증 데이터셋이 학습 과정에서 어떤 결정(조기 종료 시점 등)에 사용되는지 떠올려 보자.

### 풀이 해설

핵심은 **검증 데이터셋도 사실상 학습에 관여한다**는 점이다.

검증 데이터셋은 모델의 파라미터를 직접 바꾸지는 않는다. 역전파도 최적화도 훈련 데이터로만 이루어진다.
그러나 4-1절에서 본 조기 종료를 떠올려 보자. **검증 손실이 최소였던 시점의 파라미터로 모델을 복원한다.**
즉 '어느 시점의 모델을 최종 모델로 삼을 것인가'라는 결정을 검증 데이터셋이 내린다.

조기 종료뿐이 아니다. 학습률을 얼마로 할지, 은닉층을 몇 개로 할지, 참을성 한계를 얼마로 둘지 같은
**하이퍼파라미터 선택도 결국 검증 성능을 보고 내리는 결정**이다.
연습 문제 4-7처럼 여러 구조를 실험해 가장 좋은 것을 고르는 과정이 바로 그것이다.

이런 선택을 반복하다 보면, 파라미터는 아니지만 **모델이 검증 데이터셋에 맞춰지는 현상**이 생긴다.
여러 후보 중 검증 성능이 가장 좋은 것을 골랐다는 사실 자체가 검증 데이터셋의 정보를 모델에 흘려 넣은 셈이다.
그래서 검증 성능은 이미 낙관적으로 부풀려져 있고, 공정한 성능 지표가 될 수 없다.

본문 p5의 비유를 빌리면 이렇다. 검증 데이터셋은 **모의고사**다. 모의고사 성적을 보고 공부 방법을 바꾼다.
그런데 모의고사를 여러 번 보며 거기에 맞춰 공부했다면, 그 모의고사 점수는 더 이상 실력의 척도가 아니다.
실력을 재려면 **한 번도 보지 않은 문제**가 필요하다. 그것이 평가 데이터셋, 곧 실전 시험이다.

그래서 본문 p5는 평가 데이터셋을 "파라미터 최적화에도 학습 진행 상황 검증에도 사용하지 않은 또 다른 데이터"로
정의하고, 요점 정리에서 "모델 학습 과정에서 완전히 격리해 사용해야 한다"고 못 박는다.

### 문제 검토

- **적절성: 적합.** 본문이 세 데이터셋의 역할을 설명하긴 했지만 '왜 굳이 둘로 나누는가'를 직접 묻지는 않았다.
  이 문제가 그 빈틈을 정확히 겨눈다. 힌트도 '조기 종료 시점'이라는 구체적 사례를 짚어 줘서,
  독자가 4-1절 내용을 되짚으며 스스로 답에 도달할 수 있다.
- **[검토] 4-7과의 연결을 덧붙이면 좋다.** 검증 데이터셋이 관여하는 결정에는 조기 종료뿐 아니라
  **하이퍼파라미터 선택**도 있는데, 힌트가 조기 종료만 언급한다. 연습 문제 4-7이 바로 여러 구조를 실험해
  가장 좋은 것을 고르는 문제이므로, 힌트에 한 구절 덧붙이면 두 문제가 이어진다.

**윤문안 (힌트)**

> 힌트: 검증 데이터셋이 학습 과정에서 어떤 결정(조기 종료 시점, 하이퍼파라미터 선택 등)에 사용되는지 떠올려 보자.

## 연습 문제 4-5 [도전 문제]

> (앞부분 생략) 이런 검증 방법을 **교차 검증**이라고 하며, 위에서 소개한 방법은 대표적인 교차 검증법인
> **K-겹 교차 검증법**이다. (…) 900개의 샘플로 구성된 회오리 모양 데이터를 사용해 K=5인 K-겹 교차 검증법으로
> 모델을 학습하고 모델의 최종 분류 성능을 계산해 보자.

In [2]:
class SpiralDataset(Dataset):
    """본문 [코드 4-7]과 같은 회오리 데이터셋"""
    def __init__(self, file_path):
        with open(file_path, 'r') as f:
            rows = list(csv.DictReader(f))
        self.X = torch.tensor([[float(r['x1']), float(r['x2'])] for r in rows])
        self.Y = torch.tensor([int(r['label']) for r in rows])

    def __len__(self):
        return len(self.Y)

    def __getitem__(self, idx):
        return self.X[idx], self.Y[idx]

dataset = SpiralDataset(f'{DATA_DIR}/ch3_spiral_data.csv')

# 1) 평가 데이터셋을 먼저 떼어 내 치워 둔다
TEST_SIZE = 180
generator = torch.Generator().manual_seed(SEED)
rest_set, test_set = random_split(dataset, [len(dataset) - TEST_SIZE, TEST_SIZE],
                                  generator=generator)
print(f'교차 검증용 {len(rest_set)}개, 평가용 {len(test_set)}개')

교차 검증용 720개, 평가용 180개


In [3]:
K = 5
EPOCHS = 300
BATCH_SIZE = 100

# 2) 남은 데이터셋을 K개의 부분 데이터셋으로 나눈다
fold_sizes = [len(rest_set) // K] * K
generator = torch.Generator().manual_seed(SEED)
folds = random_split(rest_set, fold_sizes, generator=generator)
print(f'{K}개의 부분 데이터셋: 각 {fold_sizes[0]}개')

def build_model():
    return nn.Sequential(
        nn.Linear(2, HIDDEN_DIM), nn.ReLU(),
        nn.Linear(HIDDEN_DIM, HIDDEN_DIM), nn.ReLU(),
        nn.Linear(HIDDEN_DIM, 3),
    )

def train_one(train_loader, valid_loader, epochs=EPOCHS):
    torch.manual_seed(SEED)
    model = build_model()
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=LR)
    for _ in range(epochs):
        model.train()
        for inputs, labels in train_loader:
            optimizer.zero_grad()
            criterion(model(inputs), labels).backward()
            optimizer.step()
    model.eval()
    total_loss, total_correct, total = 0., 0, 0
    with torch.no_grad():
        for inputs, labels in valid_loader:
            outputs = model(inputs)
            total_loss += criterion(outputs, labels).item() * len(labels)
            total_correct += (outputs.argmax(dim=-1) == labels).sum().item()
            total += len(labels)
    return model, total_loss / total, total_correct / total * 100

5개의 부분 데이터셋: 각 144개


In [4]:
# 3) 검증 데이터셋을 바꿔 가며 K번 학습한다
models, valid_losses, valid_accuracies = [], [], []
print(f'{"회차":>4} {"검증 폴드":>8} {"검증 손실":>10} {"검증 정확도":>11}')
print('-' * 40)
for k in range(K):
    valid_fold = folds[k]
    train_folds = ConcatDataset([folds[i] for i in range(K) if i != k])   # 나머지 K-1개를 합친다
    train_loader = DataLoader(train_folds, batch_size=BATCH_SIZE, shuffle=True)
    valid_loader = DataLoader(valid_fold, batch_size=BATCH_SIZE, shuffle=False)
    model, valid_loss, valid_acc = train_one(train_loader, valid_loader)
    models.append(model); valid_losses.append(valid_loss); valid_accuracies.append(valid_acc)
    print(f'{k + 1:4d} {k:8d} {valid_loss:10.4f} {valid_acc:10.2f}%')

print('-' * 40)
print(f'검증 손실 평균 {np.mean(valid_losses):.4f} (표준편차 {np.std(valid_losses):.4f})')
print(f'검증 정확도 평균 {np.mean(valid_accuracies):.2f}% (표준편차 {np.std(valid_accuracies):.2f})')

  회차    검증 폴드      검증 손실      검증 정확도
----------------------------------------


   1        0     0.2208      93.75%


   2        1     0.2474      93.75%


   3        2     0.1642      95.83%


   4        3     0.2245      97.22%


   5        4     0.1984      92.36%
----------------------------------------
검증 손실 평균 0.2110 (표준편차 0.0281)
검증 정확도 평균 94.58% (표준편차 1.72)


In [5]:
# 4) 치워 두었던 평가 데이터셋으로 최종 성능을 측정한다
test_loader = DataLoader(test_set, batch_size=BATCH_SIZE, shuffle=False)

def evaluate(model):
    model.eval()
    correct = total = 0
    with torch.no_grad():
        for inputs, labels in test_loader:
            correct += (model(inputs).argmax(dim=-1) == labels).sum().item()
            total += len(labels)
    return correct / total * 100

best_k = int(np.argmax(valid_accuracies))
print(f'검증 성능이 가장 좋은 모델: {best_k + 1}회차')
print(f'  그 모델의 평가 정확도: {evaluate(models[best_k]):.2f}%')

# 전체 데이터로 다시 학습한 모델과 비교
full_loader = DataLoader(rest_set, batch_size=BATCH_SIZE, shuffle=True)
full_model, _, _ = train_one(full_loader, test_loader)
print(f'  전체 데이터로 다시 학습한 모델의 평가 정확도: {evaluate(full_model):.2f}%')

검증 성능이 가장 좋은 모델: 4회차
  그 모델의 평가 정확도: 92.22%


  전체 데이터로 다시 학습한 모델의 평가 정확도: 91.11%


### 풀이 해설

지문의 네 단계를 그대로 코드로 옮기면 된다. 구현에서 눈여겨볼 곳은 두 가지다.

**하나는 `ConcatDataset`으로 K-1개의 폴드를 합치는 부분이다.** 지문이 친절하게 [코드 4-15]로 알려 준 대로,
`random_split()`이 나눈 조각을 다시 붙일 때 쓴다. 매 회차마다 검증용 폴드 하나를 빼고 나머지를 합쳐
훈련 데이터로 삼는다.

**다른 하나는 모델을 회차마다 새로 만들어야 한다는 점이다.** 같은 모델을 이어서 학습하면 앞 회차의 검증 폴드를
이미 학습한 상태가 되어 교차 검증의 의미가 사라진다. 옵티마이저도 마찬가지다(3장 p33 참고).

결과에서 읽을 것은 **평균만이 아니라 표준편차**다. 다섯 번의 검증 정확도가 얼마나 흩어져 있는지가
이 성능 추정을 얼마나 믿을 수 있는지를 알려 준다. 한 번만 나눠 검증했다면 운 좋게 쉬운 검증 폴드를 만나
성능을 과대평가했을 수도 있다. K번의 평균은 그런 우연을 상당히 걷어낸다.

마지막 단계에서 '검증 성능이 가장 좋은 모델'과 '전체 데이터로 다시 학습한 모델' 중 무엇을 쓸지는 상황에 따라 다르다.
전자는 이미 학습이 끝나 있어 편하지만 데이터의 4/5만 본 모델이고, 후자는 데이터를 모두 활용하지만
그 모델 자체는 검증된 적이 없다. 교차 검증으로 얻은 성능 추정치를 그 모델의 기대 성능으로 삼는 셈이다.

실행 결과에서 한 가지가 더 눈에 띈다. **검증 정확도 평균은 94.58%인데 평가 정확도는 92.22%와 91.11%로 더 낮다.**

교차 검증으로 얻은 성능 추정치조차 실제 성능보다 낙관적이라는 뜻이다.

검증 폴드를 돌려 가며 썼어도, 그중 가장 좋은 모델을 고르는 순간 검증 데이터의 정보가 선택에 반영되기 때문이다.

[연습 문제 4-4]가 묻는 '검증 데이터셋과 평가 데이터셋을 왜 나누는가'의 답이 여기서 숫자로 확인된다.

### 문제 검토

- **적절성: 적합. 도전 문제로 잘 설계됐다.** 교차 검증은 본문에서 다루지 않은 개념인데, 지문이 네 단계로
  절차를 또박또박 제시하고 `ConcatDataset`이라는 관문까지 [코드 4-15]로 미리 풀어 준다.
  덕분에 개념은 새롭지만 구현은 4-2절에서 배운 도구만으로 가능하다. 도전 문제의 좋은 본보기다.
- **[검토] 지문이 길어 문제가 어디서 시작되는지 찾기 어렵다.** 네 단계 설명과 교차 검증의 정의, `ConcatDataset`
  사용법까지 두 쪽에 걸쳐 이어지고, 정작 해야 할 일("K=5인 K-겹 교차 검증법으로 모델을 학습하고 최종 분류
  성능을 계산해 보자")은 그 사이에 한 문장으로 들어 있다. 해야 할 일을 앞이나 뒤로 몰아 주면 읽기 쉽다.
- **[검토] 무엇을 관찰해야 하는지 없다.** K번의 검증 성능을 **평균만 내고 끝낼지, 흩어진 정도까지 볼지**에 따라
  얻는 것이 달라진다. 교차 검증의 진짜 가치는 성능 추정의 신뢰도를 함께 얻는 데 있으므로,
  표준편차나 회차별 편차를 살펴보라는 구절이 있으면 좋다.
- **[검토] 모델을 회차마다 새로 만들어야 한다는 함정.** 지문에 언급이 없어, 모델과 옵티마이저를 루프 밖에서
  한 번만 만들면 교차 검증이 무의미해진다. 3장 p33에서 옵티마이저 재사용을 다뤘으므로 힌트로 짚어 줄 만하다.

**윤문안 (마지막 문단)**

> 900개의 샘플로 구성된 회오리 모양 데이터(`data/ch3_spiral_data.csv` 파일)를 사용해 K=5인 K-겹 교차 검증법으로
> 모델을 학습하고 모델의 최종 분류 성능을 계산해 보자. K번의 검증 성능이 서로 얼마나 차이 나는지도 함께 확인해 보자.
>
> 힌트: 회차마다 모델과 옵티마이저를 새로 만들어야 한다. `random_split()` 함수와 반대로 데이터셋을 합쳐야 하는
> 경우 `torch.utils.data.ConcatDataset` 클래스를 다음과 같이 사용하면 된다.